In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# إنشاء مدير تمرير لـ Dynamical Decoupling

<Accordion>
<AccordionItem title="إصدارات الحزم">

الكود في هذه الصفحة تم تطويره باستخدام المتطلبات التالية.
نوصي باستخدام هذه الإصدارات أو أحدث منها.

```
qiskit[all]~=2.4.0
qiskit-ibm-runtime~=0.46.1
```
</AccordionItem>
</Accordion>

تُوضّح هذه الصفحة كيفية استخدام تمريرة [`PadDynamicalDecoupling`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.PadDynamicalDecoupling) لإضافة تقنية لقمع الأخطاء تُعرف بـ _Dynamical Decoupling_ إلى الدائرة.

يعمل Dynamical Decoupling عبر إضافة تسلسلات نبضات (تُعرف بـ _تسلسلات Dynamical Decoupling_) إلى الـ qubits الخاملة لتدويرها حول كرة بلوخ، مما يُلغي تأثير قنوات الضوضاء ويُقلّل من تأثير التفكك. هذه التسلسلات النبضية مشابهة لنبضات إعادة التركيز المستخدمة في الرنين المغناطيسي النووي. للاطلاع على وصف كامل، راجع [A Quantum Engineer's Guide to Superconducting qubits](https://arxiv.org/abs/1904.06560).

نظرًا لأن تمريرة `PadDynamicalDecoupling` تعمل فقط على الدوائر المجدولة زمنيًا وتتضمّن Gates ليست بالضرورة Gates أساسية لهدفنا، ستحتاج أيضًا إلى تمريرتَي [`ALAPScheduleAnalysis`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.ALAPScheduleAnalysis) و[`BasisTranslator`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.BasisTranslator).

احصل على معلومات `target` من الـ `backend` واحفظ أسماء العمليات في `basis_gates`، إذ سيحتاج `target` إلى التعديل لإضافة معلومات التوقيت للـ Gates المستخدمة في Dynamical Decoupling.

> **Note:** يُستخدم الـ Backend الوهمي `FakeSherbrooke` من `qiskit_ibm_runtime` في هذه الأمثلة، لكن يمكنك تجربته على أي Backend حقيقي أو وهمي متوافق مع Qiskit. قد تختلف نتائجك.

In [1]:
from qiskit_ibm_runtime.fake_provider import FakeSherbrooke

backend = FakeSherbrooke()

target = backend.target
basis_gates = list(target.operation_names)

أنشئ دائرة `efficient_su2` كمثال. أولًا، قم بعملية Transpile للدائرة إلى الـ Backend لأن نبضات Dynamical Decoupling تحتاج إلى إضافتها بعد أن يتم Transpile الدائرة وجدولتها زمنيًا. غالبًا ما يعمل Dynamical Decoupling بشكل أفضل عندما يكون هناك قدر كبير من وقت الخمول في الدوائر الكمومية — أي عندما لا تكون بعض الـ qubits قيد الاستخدام بينما تكون الأخرى نشطة. هذا هو الحال في هذه الدائرة لأن الـ Gates ثنائية الـ qubit من نوع `ecr` تُطبَّق بشكل تسلسلي في هذه الدالة التقريبية.

In [2]:
from qiskit.transpiler import generate_preset_pass_manager
from qiskit.circuit.library import efficient_su2

qc = efficient_su2(12, entanglement="circular", reps=1)
pm = generate_preset_pass_manager(1, target=target, seed_transpiler=12345)
qc_t = pm.run(qc)
qc_t.draw("mpl", fold=-1, idle_wires=False)

<Image src="../docs/images/guides/dynamical-decoupling-pass-manager/extracted-outputs/9479a60c-d5d0-45c7-a93e-a2a488ba8985-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/dynamical-decoupling-pass-manager/extracted-outputs/9479a60c-d5d0-45c7-a93e-a2a488ba8985-0.svg)

تسلسل Dynamical Decoupling هو سلسلة من الـ Gates تتكوّن مجتمعةً من عملية الهوية وتكون متباعدة بانتظام في الزمن. على سبيل المثال، ابدأ بإنشاء تسلسل بسيط يُسمى XY4 يتكوّن من أربع Gates.

In [3]:
from qiskit.circuit.library import XGate, YGate

X = XGate()
Y = YGate()

dd_sequence = [X, Y, X, Y]

بسبب التوقيت المنتظم لتسلسلات Dynamical Decoupling، يجب إضافة معلومات حول `YGate` إلى `target` لأنها *ليست* Gate أساسية، بعكس `XGate`. غير أننا نعلم *مسبقًا* أن `YGate` لها نفس مدة وخطأ `XGate`، لذا يمكننا فقط استرداد تلك الخصائص من `target` وإضافتها لكائنات `YGate`. هذا أيضًا هو السبب في حفظ `basis_gates` بشكل منفصل، إذ نُضيف تعليمة `YGate` إلى `target` رغم أنها ليست Gate أساسية فعلية لـ `ibm_fez`.

In [4]:
from qiskit.transpiler import InstructionProperties

y_gate_properties = {}
for qubit in range(target.num_qubits):
    y_gate_properties.update(
        {
            (qubit,): InstructionProperties(
                duration=target["x"][(qubit,)].duration,
                error=target["x"][(qubit,)].error,
            )
        }
    )

target.add_instruction(YGate(), y_gate_properties)

دوائر الدوال التقريبية مثل `efficient_su2` ذات معاملات، لذا يجب ربط قيم بها قبل إرسالها إلى الـ Backend. هنا، قم بتعيين معاملات عشوائية.

In [5]:
import numpy as np

rng = np.random.default_rng(1234)
qc_t.assign_parameters(
    rng.uniform(-np.pi, np.pi, qc_t.num_parameters), inplace=True
)

بعد ذلك، نفّذ التمريرات المخصصة. أنشئ `PassManager` بـ `ALAPScheduleAnalysis` و`PadDynamicalDecoupling`. شغّل `ALAPScheduleAnalysis` أولًا لإضافة معلومات التوقيت حول الدائرة الكمومية قبل إمكانية إضافة تسلسلات Dynamical Decoupling المتباعدة بانتظام. تُشغَّل هذه التمريرات على الدائرة باستخدام `.run()`.

In [6]:
from qiskit.transpiler import PassManager
from qiskit.transpiler.passes.scheduling import (
    ALAPScheduleAnalysis,
    PadDynamicalDecoupling,
)

dd_pm = PassManager(
    [
        ALAPScheduleAnalysis(target=target),
        PadDynamicalDecoupling(target=target, dd_sequence=dd_sequence),
    ]
)
qc_dd = dd_pm.run(qc_t)

استخدم أداة التصوير [`timeline_drawer`](https://docs.quantum.ibm.com/api/qiskit/qiskit.visualization.timeline_drawer) لرؤية توقيت الدائرة والتحقق من أن تسلسلًا منتظم التباعد من كائنات `XGate` و`YGate` يظهر في الدائرة.

In [7]:
from qiskit.visualization import timeline_drawer

timeline_drawer(qc_dd, idle_wires=False, target=target)

<Image src="../docs/images/guides/dynamical-decoupling-pass-manager/extracted-outputs/7a552621-a96f-4bb8-ae9b-4ab5a65bbb64-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/dynamical-decoupling-pass-manager/extracted-outputs/7a552621-a96f-4bb8-ae9b-4ab5a65bbb64-0.svg)

أخيرًا، بما أن `YGate` ليست Gate أساسية فعلية لـ Backend الخاص بنا، قم بتطبيق تمريرة `BasisTranslator` يدويًا (هذه تمريرة افتراضية، لكنها تُنفَّذ قبل الجدولة الزمنية، لذا تحتاج إلى تطبيقها مرة أخرى). مكتبة التكافؤ للجلسة هي مكتبة من تكافؤات الدوائر تتيح للـ Transpiler تحليل الدوائر إلى Gates أساسية، كما هو محدد أيضًا كمعامل.

In [8]:
from qiskit.circuit.equivalence_library import (
    SessionEquivalenceLibrary as sel,
)
from qiskit.transpiler.passes import BasisTranslator

qc_dd = BasisTranslator(sel, basis_gates)(qc_dd)
qc_dd.draw("mpl", fold=-1, idle_wires=False)

<Image src="../docs/images/guides/dynamical-decoupling-pass-manager/extracted-outputs/f0f4b29d-1c95-47d2-a7ad-8e130eaff74a-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/dynamical-decoupling-pass-manager/extracted-outputs/f0f4b29d-1c95-47d2-a7ad-8e130eaff74a-0.svg)

الآن، كائنات `YGate` غائبة عن دائرتنا، وتوجد معلومات توقيت صريحة في شكل Gates من نوع `Delay`. هذه الدائرة المُحوَّلة مع Dynamical Decoupling أصبحت الآن جاهزة للإرسال إلى الـ Backend.

## الخطوات التالية

> **Tip:** - لتتعلم كيفية استخدام الدالة `generate_preset_passmanager` بدلًا من كتابة تمريراتك الخاصة، ابدأ بموضوع [الإعدادات الافتراضية للـ Transpilation وخيارات الضبط](defaults-and-configuration-options).
> - جرّب دليل [مقارنة إعدادات الـ Transpiler](/guides/circuit-transpilation-settings#compare-transpiler-settings).
> - راجع [توثيق Transpile API](https://docs.quantum-computing.ibm.com/api/qiskit/transpiler).